## Python Commands for Hypothesis Testing, Regressions, and Predictions

#### 1. Import the standard libraries:
 - pandas
 - numpy
 - scipy and stats (Scientific Python is a library used for scientific and technical computing)
 - statsmodels.formula.api (a convenience module in Python's statsmodels library that lets you specify statistical models easily)

In [15]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as sf

#### 2. Bring in cleaned financial statement data

In [16]:
# function so i only type the file name (works on both computers) [I made cause I don't keep stuff in same folder]
def load_data(file_name, rows=5):
    # check the end of the file name to pick the right reader
    if file_name.endswith('.csv'):
        reader = pd.read_csv
    else:
        reader = pd.read_excel
    # try personal computer path first, then school desktop
    try:
        df = reader('C:\\Users\\Henry\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    except FileNotFoundError:
        df = reader('C:\\Users\\fbw4vt\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    # show the first rows (5 unless i give a different number)
    display(df.head(rows))
    return df

comp1 = load_data('Firm_data-1.xlsx')

,ID,year,log_assets,leverage,firm_growth,rec_inv,roa,firm_age,operating_risk,bankruptcy_risk,boardsize,acsize,big4,log_fee
0,1750,2012,7.667111,0.331602,0.939821,0.420375,0.025738,48,4,5,11,7,1,14.372629
1,1750,2013,7.695985,0.288247,0.982636,0.423187,0.033144,49,4,5,10,6,1,14.340227
2,1750,2014,7.323171,0.101650,1.132868,0.526601,0.006733,50,5,4,10,6,1,14.400165
3,1750,2015,7.273856,0.102697,0.983851,0.559185,0.033077,51,4,3,11,7,1,14.464897
4,1750,2016,7.315950,0.104581,1.190233,0.566784,0.037564,52,5,3,12,6,1,14.433289


#### 3. Hypothesis testing
Compare whether the means of two groups are statistically different (Welch's T-Test).
- step 1: specify groups based on a variable.
- step 2: run the test and save the results.
- step 3: print the results.

In [17]:
# split audit fees into big 4 vs non-big 4 firms
g_big4 = comp1[comp1['big4']==1]['log_fee']
g_nonbig4 = comp1[comp1['big4']==0]['log_fee']

# average log fee for each group
mean_big4 = np.mean(g_big4)
mean_nonbig4 = np.mean(g_nonbig4)

# print both means and the difference
print(mean_big4)
print(mean_nonbig4)
print(mean_big4 - mean_nonbig4)

14.466290466417854
12.76859049313538
1.6976999732824751


In [18]:
# welch t-test (unequal variances): small p-value means the fee difference is significant
t_stat,p_value = stats.ttest_ind(g_big4,g_nonbig4,equal_var=False)
print(p_value)

0.0


#### 4. Estimate a linear regression
**Model:** log_fee = b0 + b1 big4 + e

**Mode2:** log_fee = b0 + b1 big4 + b2 log_assets + e

- step 1: specify the model (`ols` means linear model).
- step 2: run the model and save the results.
- step 3: print the results.

In [19]:
model1 = sf.ols(formula = 'log_fee ~ big4', data=comp1) #sf is statistical models library, may not be on the lab computers?
res1 = model1.fit()
res1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                log_fee   R-squared:                       0.409
Model:                            OLS   Adj. R-squared:                  0.409
Method:                 Least Squares   F-statistic:                 1.277e+04
Date:                Tue, 20 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:37:01   Log-Likelihood:                -24972.
No. Observations:               18468   AIC:                         4.995e+04
Df Residuals:                   18466   BIC:                         4.996e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     12.7686      0.013   1015.925      0.000      12.744      12.793
big4           1.6977      0.015    113.015      0.000       1.668       1.727
==============================================================================
Omnibus:                      411.562   Durbin-Watson:                   0.487
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              439.014
Skew:                           0.374   Prob(JB):                     4.67e-96
Kurtosis:                       3.110   Cond. No.                         3.42
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [20]:
model2 = sf.ols(formula = 'log_fee ~ big4 + log_assets', data=comp1) 
res2 = model2.fit()
res2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                log_fee   R-squared:                       0.789
Model:                            OLS   Adj. R-squared:                  0.789
Method:                 Least Squares   F-statistic:                 3.452e+04
Date:                Tue, 20 Oct 2026   Prob (F-statistic):               0.00
Time:                        16:37:01   Log-Likelihood:                -15460.
No. Observations:               18468   AIC:                         3.093e+04
Df Residuals:                   18465   BIC:                         3.095e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     10.6443      0.014    768.102      0.000      10.617      10.671
big4           0.5301      0.011     48.085      0.000       0.508       0.552
log_assets     0.4537      0.002    182.389      0.000       0.449       0.459
==============================================================================
Omnibus:                      190.830   Durbin-Watson:                   0.607
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              262.692
Skew:                          -0.140   Prob(JB):                     9.06e-58
Kurtosis:                       3.513   Cond. No.                         23.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

#### 5. Estimate a Logistic regression
**Model:** Pr(big4 = 1) = Logistic(b0 + b1 log_assets + b2 leverage)
- step 1: specify the model (`logit` means logistic model).
- step 2: run the model and save the results.
- step 3: print the results.

In [21]:
model3 = sf.logit(formula = 'big4 ~ log_assets + leverage', data=comp1) # only thing that changes was .ols to .logit basically (and new model) 
res3 = model3.fit()
res3.summary()


Optimization terminated successfully.
         Current function value: 0.402692
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                   big4   No. Observations:                18468
Model:                          Logit   Df Residuals:                    18465
Method:                           MLE   Df Model:                            2
Date:                Tue, 20 Oct 2026   Pseudo R-squ.:                  0.3408
Time:                        16:37:01   Log-Likelihood:                -7436.9
converged:                       True   LL-Null:                       -11281.
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.8555      0.085    -56.864      0.000      -5.023      -4.688
log_assets     0.9918      0.015     64.298      0.000       0.962       1.022
leverage      -0.6257      0.085     -7.354      0.000      -0.792      -0.459
==============================================================================
"""

In [22]:
model3b = sf.logit(formula = 'big4 ~ log_assets + leverage', data=comp1).fit()
model3b.summary() # alternative, shorter way to do this

Optimization terminated successfully.
         Current function value: 0.402692
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                   big4   No. Observations:                18468
Model:                          Logit   Df Residuals:                    18465
Method:                           MLE   Df Model:                            2
Date:                Tue, 20 Oct 2026   Pseudo R-squ.:                  0.3408
Time:                        16:37:01   Log-Likelihood:                -7436.9
converged:                       True   LL-Null:                       -11281.
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.8555      0.085    -56.864      0.000      -5.023      -4.688
log_assets     0.9918      0.015     64.298      0.000       0.962       1.022
leverage      -0.6257      0.085     -7.354      0.000      -0.792      -0.459
==============================================================================
"""

#### 6. Run regressions by group
Much easier than building a macro in Alteryx!

In [23]:
# def groupreg(data,formula):
#     model = sf.ols(formula, data=data).fit()
#     return model.params

def groupreg(data, formula):
    model = sf.ols(formula, data=data).fit()
    return pd.Series({
        'big4_coef': model.params.get('big4'),
        'big4_pvalue': model.pvalues.get('big4'),
        'log_assets_coef': model.params.get('log_assets'),
        'log_assets_pvalue': model.pvalues.get('log_assets'),
        'n': model.nobs,
        'r2': model.rsquared
    })

group_results = (
    comp1.groupby('year')
         .apply(groupreg, "log_fee ~ big4 + log_assets", include_groups=False)
         .reset_index()
)

print(group_results.round(5)) # i added .round(5) so that its easier to see

   year  big4_coef  big4_pvalue  log_assets_coef  log_assets_pvalue       n  \
0  2012    0.48387          0.0          0.45706                0.0  2293.0   
1  2013    0.54410          0.0          0.45609                0.0  2324.0   
2  2014    0.56108          0.0          0.44392                0.0  2405.0   
3  2015    0.60771          0.0          0.44127                0.0  2396.0   
4  2016    0.53273          0.0          0.45050                0.0  2317.0   
5  2017    0.57442          0.0          0.44690                0.0  2263.0   
6  2018    0.53682          0.0          0.45174                0.0  2250.0   
7  2019    0.50901          0.0          0.45239                0.0  2220.0   

        r2  
0  0.78207  
1  0.79294  
2  0.77936  
3  0.78948  
4  0.79296  
5  0.79715  
6  0.79613  
7  0.79846  


#### 7. Predictions 1 - Big 4 Choice
**Model:** Pr(big4 = 1) = Logistic(b0 + b1 log_assets + b2 leverage)
- step 1: load new firm data.
- setp 2: estimate the model, and add .fit() at the end!
- step 3: make prediction using estimated model.
- step 4: print the results.

In [24]:
comp2 = load_data('New_firm_data-1.xlsx')

,ID,year,log_assets,leverage,firm_growth,rec_inv,roa,firm_age,operating_risk,bankruptcy_risk,boardsize,acsize
0,1,2020,3.25,0.36,1.23,0.42,0.03,6,6,6,5,3
1,2,2020,7.56,0.27,0.77,0.55,-0.05,6,5,3,10,3
2,3,2020,11.38,0.55,1.11,0.33,0.01,21,1,2,10,6


In [27]:
model3b = sf.logit(formula = 'big4 ~ log_assets + leverage', data=comp1).fit() # don't have to do this again if run above

# adding the prediction to the original (training) data
comp2['prediction'] = model3b.predict(comp2) 

# creates an indicator like we did in alteryx
comp2['big4'] = (comp2['prediction'] > 0.5).astype(int)
comp2.head()

Optimization terminated successfully.
         Current function value: 0.402692
         Iterations 7


,ID,year,log_assets,leverage,firm_growth,rec_inv,roa,firm_age,operating_risk,bankruptcy_risk,boardsize,acsize,prediction,big4
0,1,2020,3.25,0.36,1.23,0.42,0.03,6,6,6,5,3,0.135005,0
1,2,2020,7.56,0.27,0.77,0.55,-0.05,6,5,3,10,3,0.922262,1
2,3,2020,11.38,0.55,1.11,0.33,0.01,21,1,2,10,6,0.997733,1


#### 8. Prediction 2 - Audit Fees
To further predict audit fees, we will need the predicted Big 4 choices. Thus, we take the following steps:
- step 1: Append the predicted probabilities to comp2
- step 2: Determine Big 4 choices
- step 3: Estimate the linear model (again, add .fit() at the end! ) and make predictions
- step 4: Print the results

In [ ]:
# we did steps 1 & 2 above

# step 3
model2 = sf.ols(formula = 'log_fee ~ big4 + log_assets', data=comp1).fit()

prediction2 = model2.predict(comp2)

# step 4
print(prediction2)

0    12.118690
1    14.604069
2    16.337055
dtype: float64
